# Baseline Reliability and Comparative Evaluation

**Purpose.** Validate the repository policy against the official simulator,
then compare it with Kaggle's random sample policy across both player seats.

**Decision question.** Is the deterministic policy reliable, and does it
provide measurable improvement over the official control? This is an
offline screening result, not a ladder-rating estimate.

## 1. Configuration and reproducibility limits

The previous run completed four self-play games but exposed only numeric
context IDs and no control comparison. This revision records named decision
contexts and action types, retains every failure, and balances candidate
games across both seats.

Python's random policy is seeded. The current simulator wrapper does not
expose its internal card-draw or coin-toss seed, so seat-balanced games are
independent repetitions rather than exact paired seeds.

In [ ]:
from collections import Counter
from pathlib import Path
import importlib.util
import json
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd

CONTRACT_GAMES = 4
GAMES_PER_SEAT = 20
MAX_DECISIONS = 10_000
BASE_SEED = 42
BOOTSTRAP_SAMPLES = 10_000
WORK_DIR = Path("/kaggle/working/agent_eval")

In [ ]:
def first_match(pattern: str) -> Path:
    matches = sorted(Path("/kaggle/input").rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No Kaggle input matched {pattern}")
    return matches[0]

def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

sample_dir = first_match("sample_submission/main.py").parent
candidates = [Path("../agent"), Path("agent")]
candidates += [
    path.parent for path in sorted(Path("/kaggle/input").rglob("main.py"))
    if "sample_submission" not in path.parts and "cg" not in path.parts
]
repo_agent = next(
    (path for path in candidates
     if (path / "main.py").exists() and (path / "deck.csv").exists()),
    None,
)
if repo_agent is None:
    raise FileNotFoundError("Attach the private agent-source dataset.")
print(f"Official control: {sample_dir / 'main.py'}")
print(f"Candidate: {repo_agent / 'main.py'}")

## 2. Isolated simulator and policies

The complete official `cg` directory is copied into working storage. The
candidate and control are loaded as separate modules, while both use the
same reviewed 60-card deck so this experiment isolates policy behavior.

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(sample_dir, WORK_DIR)
shutil.copy2(repo_agent / "main.py", WORK_DIR / "candidate_main.py")
shutil.copy2(repo_agent / "deck.csv", WORK_DIR / "deck.csv")

sys.path.insert(0, str(WORK_DIR))
from cg.api import OptionType, SelectContext, to_observation_class
from cg.game import battle_finish, battle_select, battle_start

candidate = load_module("candidate_policy", WORK_DIR / "candidate_main.py")
control = load_module("official_random_policy", sample_dir / "main.py")
deck = candidate.read_deck_csv()
assert len(deck) == 60

## 3. Instrumented game runner

Every action is checked before the simulator receives it. Telemetry uses
enum names rather than opaque integers, making high-frequency decision
bottlenecks immediately visible. Exceptions and decision-limit stalls are
preserved as failed games.

In [ ]:
def enum_name(enum_class, value) -> str:
    try:
        return enum_class(value).name
    except (ValueError, TypeError):
        return f"UNKNOWN_{value}"

def validate_action(obs, action: list[int]) -> None:
    select = obs.select
    assert isinstance(action, list)
    assert all(isinstance(index, int) for index in action)
    assert len(action) == len(set(action))
    assert select.minCount <= len(action) <= select.maxCount
    assert all(0 <= index < len(select.option) for index in action)

def play_game(
    policies: dict[int, object],
    game_id: int,
    candidate_player: int | None,
    experiment: str,
) -> dict:
    random.seed(BASE_SEED + game_id)
    started = time.perf_counter()
    decisions = 0
    contexts, actions = Counter(), Counter()
    role_contexts, role_actions = Counter(), Counter()
    try:
        obs_dict, start_data = battle_start(deck, deck)
        if obs_dict is None:
            return {
                "status": "start_error", "experiment": experiment,
                "game": game_id, "candidate_player": candidate_player,
                "error_player": start_data.errorPlayer,
                "error_type": start_data.errorType,
            }
        while decisions < MAX_DECISIONS:
            obs = to_observation_class(obs_dict)
            if obs.current is not None and obs.current.result != -1:
                winner = int(obs.current.result)
                if candidate_player is None:
                    candidate_score = None
                elif winner == candidate_player:
                    candidate_score = 1.0
                elif winner in (0, 1):
                    candidate_score = 0.0
                else:
                    candidate_score = 0.5
                return {
                    "status": "finished", "experiment": experiment,
                    "game": game_id, "candidate_player": candidate_player,
                    "winner": winner, "candidate_score": candidate_score,
                    "turn": int(obs.current.turn), "decisions": decisions,
                    "seconds": time.perf_counter() - started,
                    "contexts": dict(contexts), "actions": dict(actions),
                    "role_contexts": dict(role_contexts),
                    "role_actions": dict(role_actions),
                }
            player = int(obs.current.yourIndex)
            role = (
                "candidate"
                if candidate_player is None or player == candidate_player
                else "control"
            )
            context_name = enum_name(SelectContext, obs.select.context)
            contexts[context_name] += 1
            role_contexts[f"{role}:{context_name}"] += 1
            action = policies[player].agent(obs_dict)
            validate_action(obs, action)
            for index in action:
                action_name = enum_name(OptionType, obs.select.option[index].type)
                actions[action_name] += 1
                role_actions[f"{role}:{action_name}"] += 1
            obs_dict = battle_select(action)
            decisions += 1
        return {
            "status": "decision_limit", "experiment": experiment,
            "game": game_id, "candidate_player": candidate_player,
            "decisions": decisions,
        }
    except Exception as error:
        return {
            "status": "exception", "experiment": experiment,
            "game": game_id, "candidate_player": candidate_player,
            "error": f"{type(error).__name__}: {error}",
            "decisions": decisions,
        }
    finally:
        try:
            battle_finish()
        except Exception:
            pass

## 4. Contract smoke test

Candidate-versus-candidate games test deck initialization, action legality,
termination, and runtime. They do not measure strength.

In [ ]:
contract_results = [
    play_game({0: candidate, 1: candidate}, game, None, "contract_self_play")
    for game in range(CONTRACT_GAMES)
]
contract_df = pd.DataFrame(contract_results)
display(contract_df.drop(
    columns=["contexts", "actions", "role_contexts", "role_actions"],
    errors="ignore",
))
contract_failures = contract_df[contract_df.status != "finished"]
assert contract_failures.empty, contract_failures.to_dict("records")
print(f"PASS: {len(contract_df)}/{len(contract_df)} contract games finished")

## 5. Candidate versus official random control

The candidate plays an equal number of games as player 0 and player 1.
A bootstrap interval summarizes game-level uncertainty. Because simulator
seeds are unavailable, treat this as a screening estimate rather than a
paired causal measurement.

In [ ]:
matchup_results = []
game_id = 10_000
for candidate_player in (0, 1):
    for repetition in range(GAMES_PER_SEAT):
        policies = {
            candidate_player: candidate,
            1 - candidate_player: control,
        }
        matchup_results.append(play_game(
            policies, game_id, candidate_player, "candidate_vs_random"
        ))
        game_id += 1

matchup_df = pd.DataFrame(matchup_results)
display(matchup_df.drop(
    columns=["contexts", "actions", "role_contexts", "role_actions"],
    errors="ignore",
))
failures = matchup_df[matchup_df.status != "finished"]
finished = matchup_df[matchup_df.status == "finished"].copy()
assert failures.empty, failures.to_dict("records")

scores = finished.candidate_score.to_numpy(dtype=float)
rng = np.random.default_rng(BASE_SEED)
bootstrap_means = rng.choice(
    scores, size=(BOOTSTRAP_SAMPLES, len(scores)), replace=True
).mean(axis=1)
ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
wins = int((scores == 1.0).sum())
draws = int((scores == 0.5).sum())
losses = int((scores == 0.0).sum())
summary = {
    "games": len(finished), "wins": wins, "draws": draws, "losses": losses,
    "score_rate": float(scores.mean()),
    "bootstrap_95_low": float(ci_low), "bootstrap_95_high": float(ci_high),
    "failures": len(failures),
}
if len(failures):
    decision = "REJECT: runtime failures observed"
elif ci_high < 0.5:
    decision = "REJECT: candidate is worse than random control"
elif ci_low > 0.5:
    decision = "PASS SCREEN: evaluate against stronger frozen controls"
else:
    decision = "HOLD: interval overlaps parity"
summary["decision"] = decision
display(pd.Series(summary).to_frame("value"))
print(f"Promotion decision: {decision}")
display(finished.groupby("candidate_player").candidate_score.agg(
    games="size", score_rate="mean"
))

## 6. Decision-context and action telemetry

The first run spent most decisions in context `0`, which was unreadable.
Named aggregation below shows where future heuristics or search can affect
the largest share of decisions and which actions the policy actually uses.

In [ ]:
context_counts, action_counts = Counter(), Counter()
role_context_counts, role_action_counts = Counter(), Counter()
for row in contract_results + matchup_results:
    context_counts.update(row.get("contexts", {}))
    action_counts.update(row.get("actions", {}))
    role_context_counts.update(row.get("role_contexts", {}))
    role_action_counts.update(row.get("role_actions", {}))

context_df = pd.Series(context_counts, name="decisions").sort_values(ascending=False).to_frame()
action_df = pd.Series(action_counts, name="selections").sort_values(ascending=False).to_frame()
display(context_df)
display(action_df)

role_action_rows = [
    {"role": key.split(":", 1)[0], "action": key.split(":", 1)[1],
     "selections": value}
    for key, value in role_action_counts.items()
]
role_context_rows = [
    {"role": key.split(":", 1)[0], "context": key.split(":", 1)[1],
     "decisions": value}
    for key, value in role_context_counts.items()
]
role_action_df = pd.DataFrame(role_action_rows).pivot_table(
    index="action", columns="role", values="selections", fill_value=0
).sort_values("candidate", ascending=False)
role_context_df = pd.DataFrame(role_context_rows).pivot_table(
    index="context", columns="role", values="decisions", fill_value=0
).sort_values("candidate", ascending=False)
display(role_action_df)
display(role_context_df)
display(finished[["turn", "decisions", "seconds"]].describe().T)

output = Path("/kaggle/working")
payload = {
    "configuration": {
        "contract_games": CONTRACT_GAMES,
        "games_per_seat": GAMES_PER_SEAT,
        "simulator_seed_exposed": False,
    },
    "summary": summary,
    "context_counts": dict(context_counts),
    "action_counts": dict(action_counts),
    "role_context_counts": dict(role_context_counts),
    "role_action_counts": dict(role_action_counts),
    "contract_results": contract_results,
    "matchup_results": matchup_results,
}
(output / "agent_evaluation_results.json").write_text(
    json.dumps(payload, indent=2, default=str)
)
print(f"Saved evaluation evidence to {output / 'agent_evaluation_results.json'}")

## 7. Promotion decision

Reliability is mandatory. Strength promotion additionally requires a score
rate above 0.5 with an uncertainty interval that is useful for the decision,
no material seat collapse, and zero runtime failures. This random-policy
comparison is only the first control; the next notebook version should add
frozen historical and strategy-diverse opponents before a ladder submission.